In [1]:
library(readr)
df <- read_csv("mxmh_survey_results.csv")
head(df)

Rows: 736 Columns: 33
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (26): Timestamp, Primary streaming service, While working, Instrumentali...
dbl  (7): Age, Hours per day, BPM, Anxiety, Depression, Insomnia, OCD

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Timestamp,Age,Primary streaming service,Hours per day,While working,Instrumentalist,Composer,Fav genre,Exploratory,Foreign languages,⋯,Frequency [R&B],Frequency [Rap],Frequency [Rock],Frequency [Video game music],Anxiety,Depression,Insomnia,OCD,Music effects,Permissions
<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
8/27/2022 19:29:02,18,Spotify,3.0,Yes,Yes,Yes,Latin,Yes,Yes,⋯,Sometimes,Very frequently,Never,Sometimes,3,0,1,0,NA,I understand.
8/27/2022 19:57:31,63,Pandora,1.5,Yes,No,No,Rock,Yes,No,⋯,Sometimes,Rarely,Very frequently,Rarely,7,2,2,1,NA,I understand.
8/27/2022 21:28:18,18,Spotify,4.0,No,No,No,Video game music,No,Yes,⋯,Never,Rarely,Rarely,Very frequently,7,7,10,2,No effect,I understand.
8/27/2022 21:40:40,61,YouTube Music,2.5,Yes,No,Yes,Jazz,Yes,Yes,⋯,Sometimes,Never,Never,Never,9,7,3,3,Improve,I understand.
8/27/2022 21:54:47,18,Spotify,4.0,Yes,No,No,R&B,Yes,No,⋯,Very frequently,Very frequently,Never,Rarely,7,2,5,9,Improve,I understand.
8/27/2022 21:56:50,18,Spotify,5.0,Yes,Yes,Yes,Jazz,Yes,Yes,⋯,Very frequently,Very frequently,Very frequently,Never,8,8,7,7,Improve,I understand.


In [2]:
summary(df$OCD)
# Расчёт моды
library(modeest)
mode_value <- mfv(df$OCD)
print(paste("Мода:", mode_value))
# Дисперсия и стандартное отклонение
variance <- var(df$OCD)
std_dev <- sd(df$OCD)
print(paste("Дисперсия:", variance))
print(paste("Стандартное отклонение:", std_dev))

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   0.000   2.000   2.637   5.000  10.000 

[1] "Мода: 0"
[1] "Дисперсия: 8.07706115054718"
[1] "Стандартное отклонение: 2.8420170918816"


In [3]:
# Загрузка необходимых библиотек
library(dplyr)

# Гипотезы
h0 <- "Любимый жанр людей с повышенной тревожностью - Рок"
h1 <- "Любимый жанр людей с повышенной тревожностью - не Рок, а Поп музыка"
alpha <- 0.05

# Выбор групп для анализа
group1 <- df %>% filter(`Fav genre` == "Rock") %>% pull(Anxiety)
group2 <- df %>% filter(`Fav genre` == "Pop") %>% pull(Anxiety)

# T-тест
t_test_result <- t.test(group1, group2, alternative = "two.sided")
t_stat <- t_test_result$statistic
p_value_t <- t_test_result$p.value

h0_test <- "Средние уровни тревожности в обеих группах равны"
h1_test <- "Средние уровни тревожности в обеих группах отличаются"

cat("t-статистика:", t_stat, "p-значение:", p_value_t, "\n")
if (p_value_t >= alpha) {
  cat(h0_test, "\n")
} else {
  cat(h1_test, "\n")
}

if (mean(group1) > mean(group2)) {
  cat(h0, "\n")
} else {
  cat(h1, "\n")
}



Присоединяю пакет: 'dplyr'


Следующие объекты скрыты от 'package:stats':

    filter, lag


Следующие объекты скрыты от 'package:base':

    intersect, setdiff, setequal, union




t-статистика: 0.1556923 p-значение: 0.8763912 
Средние уровни тревожности в обеих группах равны 
Любимый жанр людей с повышенной тревожностью - Рок 


In [5]:
# U-критерий Манна-Уитни
u_test_result <- wilcox.test(group1, group2, alternative = "two.sided")
u_stat <- u_test_result$statistic
p_value_u <- u_test_result$p.value

cat("U-статистика:", u_stat, "p-значение:", p_value_u, "\n")
if (p_value_u >= alpha) {
  cat(h0_test, "\n")
} else {
  cat(h1_test, "\n")
}



U-статистика: 11329.5 p-значение: 0.4010276 
Средние уровни тревожности в обеих группах равны 


In [6]:
# Хи-квадрат тест
# Определение медианы уровня тревожности
anxiety_median <- median(df$Anxiety)

# Создание новой категориальной переменной
df$Anxiety_Category <- ifelse(df$Anxiety < anxiety_median, "Low Anxiety", "High Anxiety")

# Фильтрация датасета
filtered_df <- df %>% filter(`Fav genre` %in% c("Rock", "Pop"))

# Создание таблицы сопряжённости
contingency_table <- table(filtered_df$`Fav genre`, filtered_df$Anxiety_Category)

# Выполнение теста хи-квадрат
chi2_test_result <- chisq.test(contingency_table)
chi2_stat <- chi2_test_result$statistic
p_value_chi2 <- chi2_test_result$p.value

cat("Хи-квадрат:", chi2_stat, "p-значение:", p_value_chi2, "\n")
if (p_value_chi2 >= alpha) {
  cat(h0_test, "\n")
} else {
  cat(h1_test, "\n")
}

Хи-квадрат: 0.0801451 p-значение: 0.7771009 
Средние уровни тревожности в обеих группах равны 
